# llm-safe-pl — anonymize Polish PII before sending documents to an LLM

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Tatarinho/llm-safe-pl/blob/main/notebooks/quickstart.ipynb)
[![PyPI version](https://img.shields.io/pypi/v/llm-safe-pl.svg)](https://pypi.org/project/llm-safe-pl/)
[![GitHub](https://img.shields.io/badge/github-Tatarinho%2Fllm--safe--pl-blue)](https://github.com/Tatarinho/llm-safe-pl)

This notebook walks through the full round-trip that `llm-safe-pl` is built for:

1. **Detect** PII in a Polish document — PESEL, NIP, REGON, IBAN, phone, email — all checksum-validated where applicable.
2. **Anonymize** — replace each hit with a stable `[TYPE_NNN]` token and return a reversible mapping.
3. **Call an LLM** on the anonymized text so the model provider never sees raw PII.
4. **Deanonymize** — restore original values in the LLM's response using the mapping.

The LLM call uses OpenAI if `OPENAI_API_KEY` is set, and falls back to a hand-crafted response otherwise, so the notebook runs end-to-end with no API key.

In [ ]:
!pip install -q llm-safe-pl openai

## The scenario

A Polish customer-service email containing multiple identifiers. This is the kind of text you might want an LLM to summarize without exposing the underlying PII to your model provider.

In [ ]:
document = """\
Dzień dobry,

Piszę w sprawie zamówienia nr INV-2025-00412. Klient: Jan Kowalski,
PESEL 44051401359, kontakt +48 600 123 456, e-mail jan.kowalski@example.pl.
Faktura wystawiona na firmę Acme Polska Sp. z o.o., NIP 526-000-12-46,
REGON 123456785, adres: ul. Marszałkowska 1, 00-001 Warszawa.
Przelew na IBAN PL61 1090 1014 0000 0712 1981 2874 nie dotarł do 2025-03-18.

Proszę o sprawdzenie statusu i kontakt zwrotny.
Pozdrawiam,
Anna Nowak
"""

print(document)

## Step 1 — detect

`Shield.detect()` returns an ordered tuple of `Match` objects. Each match carries its type, the raw value, its character offsets, and which detector fired — useful when you want an audit trail without yet committing to rewriting the text.

In [ ]:
from llm_safe_pl import Shield

shield = Shield()
matches = shield.detect(document)

print(f"Found {len(matches)} PII hits:\n")
for m in matches:
    print(f"  [{m.type.value:<11}] {m.value!r:<40} at {m.start}–{m.end}  (detector: {m.detector})")

## Step 2 — anonymize

`Shield.anonymize()` runs the same pipeline as `detect()`, but also rewrites the text and updates the shield's `Mapping`. The same `(type, value)` pair always maps to the same token within one shield — so if `Jan Kowalski` appears in three documents, it gets the same `[PERSON_001]` across all three.

Formatted identifiers (dashes in NIP, spaces in IBAN) are preserved byte-for-byte on round-trip.

In [ ]:
result = shield.anonymize(document)

print("Anonymized — safe to send to an LLM:\n")
print(result.text)

## Step 3 — call the LLM

The anonymized text is what you would send to OpenAI, Anthropic, or any local model. Tokens like `[PESEL_001]` look like ordinary placeholders and the model will happily quote them back in its reply.

**System-prompt tip:** ask the model to keep `[TYPE_NNN]` tokens intact. Not strictly required, but it reduces the chance the model rephrases `[PESEL_001]` into something else.

If `OPENAI_API_KEY` is set in your Colab environment, the real API is called. Otherwise we simulate a plausible summary so the rest of the notebook still works.

In [ ]:
import os

SYSTEM = (
    "You are a Polish-language customer service assistant. "
    "Summarize the user's message in 3 bullet points. "
    "Keep every placeholder of the form [TYPE_NNN] intact — do not rename, "
    "translate, or expand them."
)

if os.environ.get("OPENAI_API_KEY"):
    from openai import OpenAI

    client = OpenAI()
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": SYSTEM},
            {"role": "user", "content": result.text},
        ],
    )
    llm_output = response.choices[0].message.content or ""
else:
    llm_output = (
        "Podsumowanie zgłoszenia:\n"
        "- Klient [PESEL_001] zgłasza brak przelewu dla zamówienia INV-2025-00412.\n"
        "- Kontakt zwrotny: [PHONE_001] lub [EMAIL_001].\n"
        "- Faktura VAT: [NIP_001], REGON [REGON_001], IBAN [IBAN_001] — sprawdzić status transakcji."
    )

print("LLM response (still anonymized):\n")
print(llm_output)

## Step 4 — deanonymize

`Shield.deanonymize()` uses the mapping built during `anonymize()` to put real values back. This is the step that closes the loop: the model never saw the originals, but downstream systems — a ticket, a CRM write, an outgoing email — see fully restored text.

In [ ]:
restored = shield.deanonymize(llm_output)

print("Final, de-anonymized output:\n")
print(restored)

## Persisting the mapping

Real LLM workflows are often async: you anonymize now, call the model later, and deanonymize when the response comes back — possibly from a different process. `Mapping` is JSON-serializable so it rides along with whatever queue or task record you are using.

In [ ]:
import json

from llm_safe_pl import Mapping

serialized = result.mapping.to_json()
print("Serialized mapping:\n")
print(json.dumps(json.loads(serialized), indent=2, ensure_ascii=False))

rehydrated = Mapping.from_json(serialized)
print("\nRestored via a rehydrated mapping (e.g. from a queue):\n")
print(shield.deanonymize(llm_output, mapping=rehydrated))

## What this install did — and did not — catch

The core install uses **stdlib + `typer` only**. That is enough for every identifier with a deterministic format: PESEL, NIP, REGON, ID card, passport, phone, email, IBAN, credit card. Checksum validators (PESEL, NIP, REGON, Luhn, mod-97 IBAN) filter out valid-looking-but-wrong numbers so the audit log is not flooded with false positives.

**Not caught in the core install** — these are in the document above but absent from the audit trail:

- `Jan Kowalski`, `Anna Nowak` — person names
- `Acme Polska Sp. z o.o.` — organization
- `ul. Marszałkowska 1, 00-001 Warszawa` — address

These require the optional `[ner]` extra, which pulls in spaCy and a Polish model:

```bash
pip install "llm-safe-pl[ner]"
python -m spacy download pl_core_news_lg
```

NER support ships in v0.1.1 — see the [roadmap](https://github.com/Tatarinho/llm-safe-pl#roadmap).

## Next steps

- [README](https://github.com/Tatarinho/llm-safe-pl) — full feature list and install options.
- [`docs/llm_workflow.md`](https://github.com/Tatarinho/llm-safe-pl/blob/main/docs/llm_workflow.md) — deeper guidance on the anonymize → LLM → deanonymize pattern.
- [`docs/limitations.md`](https://github.com/Tatarinho/llm-safe-pl/blob/main/docs/limitations.md) — read before shipping to production.

Found a false positive, a missed identifier, or have a feature idea? [Open an issue](https://github.com/Tatarinho/llm-safe-pl/issues). Stars welcome.